# Import necessary library

In [ ]:
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Confugration

In [ ]:
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Model_Dataset"
MODEL_DIR = os.path.join(DATA_DIR, "model_artifacts_A")
os.makedirs(MODEL_DIR, exist_ok=True)

CONFIG = {
    "data_dir": DATA_DIR,
    "model_dir": MODEL_DIR,

    "time_col": "timestamp",
    "gap_col": "diff_time",
    "label_col": "label",

    # default full set
    "feature_cols": [
        "ax", "ay", "az",
        "gx", "gy", "gz",
        "mag_acc", "mag_gyro",
        "diff_acc", "diff_gyro"
    ],

    "window_size": 7,
    "stride": 1,
    "max_gap_ms": 3000,
    "min_positive_rows_in_window": 2,

    "random_state": 42
}

FEATURE_SET_OPTIONS = {
    "A_all_10": [
        "ax", "ay", "az",
        "gx", "gy", "gz",
        "mag_acc", "mag_gyro",
        "diff_acc", "diff_gyro"
    ]
}

SEARCH_SPACE = {
    "window_size": [5, 7, 9],
    "min_positive_rows_in_window": [1, 2, 3],
    "models": [
        {
            "model_name": "RandomForest",
            "params_list": [
                {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"},
                {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 2, "max_features": "sqrt"},
                {"n_estimators": 700, "max_depth": 12, "min_samples_leaf": 2, "max_features": "sqrt"},
            ]
        },
        {
            "model_name": "ExtraTrees",
            "params_list": [
                {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "max_features": "sqrt"},
                {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 2, "max_features": "sqrt"},
                {"n_estimators": 700, "max_depth": 12, "min_samples_leaf": 2, "max_features": "sqrt"},
            ]
        }
    ]
}

print(json.dumps(CONFIG, indent=2))
print("\nFeature sets:", list(FEATURE_SET_OPTIONS.keys()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "data_dir": "/content/drive/MyDrive/Model_Dataset",
  "model_dir": "/content/drive/MyDrive/Model_Dataset/model_artifacts_search",
  "time_col": "timestamp",
  "gap_col": "diff_time",
  "label_col": "label",
  "feature_cols": [
    "ax",
    "ay",
    "az",
    "gx",
    "gy",
    "gz",
    "mag_acc",
    "mag_gyro",
    "diff_acc",
    "diff_gyro"
  ],
  "window_size": 7,
  "stride": 1,
  "max_gap_ms": 3000,
  "min_positive_rows_in_window": 2,
  "random_state": 42
}

Feature sets: ['A_all_10']


# Load CSV files

In [ ]:
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print("CSV files found:", len(csv_files))
for f in csv_files:
    print("-", os.path.basename(f))

if len(csv_files) == 0:
    raise ValueError(f"No CSV files found in {DATA_DIR}")

all_dfs = []
for fp in csv_files:
    df = pd.read_csv(fp)
    df["session_id"] = os.path.basename(fp)   # one file = one session
    all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)

print("Total raw rows:", len(df))
df.head()

CSV files found: 7
- Dataset1.csv
- Dataset2.csv
- Dataset3.csv
- Dataset4.csv
- Dataset5.csv
- Dataset6.csv
- Dataset7.csv
Total raw rows: 1967


,timestamp,ax,ay,az,gx,gy,gz,mag_acc,mag_gyro,diff_acc,diff_gyro,diff_time,label,session_id
0,965,1.54,-9.73,-2.48,0.29,0.58,0.16,10.158489,0.667907,NaN,NaN,NaN,0,Dataset1.csv
1,1966,0.09,-9.81,-0.19,-0.34,0.02,0.20,9.812253,0.394968,-0.346237,-0.272939,1001.0,0,Dataset1.csv
2,2972,-0.05,-10.00,-1.12,-0.34,-0.30,0.10,10.062649,0.464327,0.250396,0.069359,1006.0,0,Dataset1.csv
3,3973,0.33,-10.22,-3.37,0.41,-0.31,0.15,10.766346,0.535444,0.703697,0.071116,1001.0,0,Dataset1.csv
4,4979,0.42,-9.84,-2.53,-0.48,-0.43,0.13,10.168722,0.657419,-0.597624,0.121975,1006.0,0,Dataset1.csv


# Data cleaning

In [ ]:
required_cols = [
    CONFIG["time_col"],
    "ax", "ay", "az",
    "gx", "gy", "gz",
    "mag_acc", "mag_gyro",
    "diff_acc", "diff_gyro",
    "diff_time",
    "label",
    "session_id"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# convert to numeric
for col in required_cols:
    if col != "session_id":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# map label if it came as text
if df["label"].dtype == object:
    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "fall": 1,
            "not fall": 0,
            "not_fall": 0,
            "no fall": 0,
            "0": 0,
            "1": 1
        })
    )

# fill diff columns
df["diff_acc"] = df["diff_acc"].fillna(0)
df["diff_gyro"] = df["diff_gyro"].fillna(0)
df["diff_time"] = df["diff_time"].fillna(0)

# fill label
df["label"] = df["label"].fillna(0).astype(int)

# drop rows with missing core features
df = df.dropna(subset=CONFIG["feature_cols"] + [CONFIG["time_col"], "label"]).copy()

# sort
df = df.sort_values(["session_id", CONFIG["time_col"]]).reset_index(drop=True)

print("Clean rows:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts())

print("\nRows per file:")
print(df.groupby("session_id").size())

print("\nPositive rows per file:")
print(df.groupby("session_id")["label"].sum())

Clean rows: 1967

Label counts:
label
0    1618
1     349
Name: count, dtype: int64

Rows per file:
session_id
Dataset1.csv    239
Dataset2.csv    118
Dataset3.csv    221
Dataset4.csv    358
Dataset5.csv    160
Dataset6.csv    798
Dataset7.csv     73
dtype: int64

Positive rows per file:
session_id
Dataset1.csv      0
Dataset2.csv     17
Dataset3.csv     56
Dataset4.csv     64
Dataset5.csv     15
Dataset6.csv    186
Dataset7.csv     11
Name: label, dtype: int64


# Create chunk_id
   Split when:
  - session changes
  - or diff_time is too large

In [ ]:
df["new_chunk"] = (
    (df["session_id"] != df["session_id"].shift(1)) |
    (df[CONFIG["gap_col"]] > CONFIG["max_gap_ms"]) |
    (df[CONFIG["gap_col"]] < 0)
).astype(int)

df["chunk_id"] = df["new_chunk"].cumsum()

print("Total chunks:", df["chunk_id"].nunique())
df[[CONFIG["time_col"], "session_id", "diff_time", "label", "chunk_id"]].head(20)

Total chunks: 101


,timestamp,session_id,diff_time,label,chunk_id
0,965,Dataset1.csv,0.0,0,1
1,1966,Dataset1.csv,1001.0,0,1
2,2972,Dataset1.csv,1006.0,0,1
3,3973,Dataset1.csv,1001.0,0,1
4,4979,Dataset1.csv,1006.0,0,1
5,5980,Dataset1.csv,1001.0,0,1
6,6986,Dataset1.csv,1006.0,0,1
7,7987,Dataset1.csv,1001.0,0,1
8,8993,Dataset1.csv,1006.0,0,1
9,9994,Dataset1.csv,1001.0,0,1


# Inspect label

In [ ]:
def get_positive_segments(group_df, time_col="timestamp", label_col="label"):
    labels = group_df[label_col].values
    ts = group_df[time_col].values
    segments = []

    in_seg = False
    start_idx = None

    for i, lab in enumerate(labels):
        if lab == 1 and not in_seg:
            in_seg = True
            start_idx = i

        if in_seg and (i == len(labels)-1 or labels[i+1] != 1):
            end_idx = i
            segments.append({
                "start_ts": int(ts[start_idx]),
                "end_ts": int(ts[end_idx]),
                "rows": int(end_idx - start_idx + 1)
            })
            in_seg = False

    return segments

for sid, g in df.groupby("session_id"):
    segs = get_positive_segments(g)
    print(f"\n{sid} -> {len(segs)} positive segment(s)")
    print(segs[:10])


Dataset1.csv -> 0 positive segment(s)
[]

Dataset2.csv -> 5 positive segment(s)
[{'start_ts': 12000, 'end_ts': 16014, 'rows': 5}, {'start_ts': 47115, 'end_ts': 50123, 'rows': 4}, {'start_ts': 83232, 'end_ts': 86240, 'rows': 4}, {'start_ts': 126375, 'end_ts': 127381, 'rows': 2}, {'start_ts': 154462, 'end_ts': 155468, 'rows': 2}]

Dataset3.csv -> 7 positive segment(s)
[{'start_ts': 16010, 'end_ts': 18017, 'rows': 3}, {'start_ts': 41095, 'end_ts': 57148, 'rows': 14}, {'start_ts': 82232, 'end_ts': 96280, 'rows': 14}, {'start_ts': 127390, 'end_ts': 131403, 'rows': 4}, {'start_ts': 162498, 'end_ts': 168519, 'rows': 7}, {'start_ts': 208655, 'end_ts': 216683, 'rows': 9}, {'start_ts': 241767, 'end_ts': 253801, 'rows': 5}]

Dataset4.csv -> 15 positive segment(s)
[{'start_ts': 15008, 'end_ts': 19021, 'rows': 4}, {'start_ts': 43097, 'end_ts': 46105, 'rows': 4}, {'start_ts': 59150, 'end_ts': 64164, 'rows': 5}, {'start_ts': 111312, 'end_ts': 123346, 'rows': 5}, {'start_ts': 144414, 'end_ts': 160461

# Feature extraction

In [ ]:
def window_to_feature_vector(window_df, feature_cols):
    feats = {}

    for col in feature_cols:
        x = window_df[col].values.astype(float)

        feats[f"{col}_mean"] = float(np.mean(x))
        feats[f"{col}_std"] = float(np.std(x))
        feats[f"{col}_median"] = float(np.median(x))
        feats[f"{col}_min"] = float(np.min(x))
        feats[f"{col}_max"] = float(np.max(x))
        feats[f"{col}_range"] = float(np.max(x) - np.min(x))
        feats[f"{col}_absmax"] = float(np.max(np.abs(x)))
        feats[f"{col}_delta"] = float(x[-1] - x[0])
        feats[f"{col}_rms"] = float(np.sqrt(np.mean(np.square(x))))
        feats[f"{col}_iqr"] = float(np.percentile(x, 75) - np.percentile(x, 25))

    return feats


def build_window_dataset(df, config, selected_feature_cols=None, window_size=None, min_positive_rows=None):
    rows = []

    feature_cols = selected_feature_cols if selected_feature_cols is not None else config["feature_cols"]
    window_size = window_size if window_size is not None else config["window_size"]
    min_positive_rows = min_positive_rows if min_positive_rows is not None else config["min_positive_rows_in_window"]
    stride = config["stride"]

    for chunk_id, g in df.groupby("chunk_id"):
        g = g.sort_values(config["time_col"]).reset_index(drop=True)

        if len(g) < window_size:
            continue

        for start in range(0, len(g) - window_size + 1, stride):
            end = start + window_size
            w = g.iloc[start:end].copy()

            positive_rows = int((w[config["label_col"]] == 1).sum())
            y = 1 if positive_rows >= min_positive_rows else 0

            feat_dict = window_to_feature_vector(w, feature_cols)

            feat_dict["label"] = y
            feat_dict["positive_rows_in_window"] = positive_rows
            feat_dict["session_id"] = w["session_id"].iloc[0]
            feat_dict["chunk_id"] = int(chunk_id)
            feat_dict["window_start_ts"] = int(w[config["time_col"]].iloc[0])
            feat_dict["window_end_ts"] = int(w[config["time_col"]].iloc[-1])

            rows.append(feat_dict)

    return pd.DataFrame(rows)

# Search best setup

In [ ]:
def make_model(model_name, params, random_state=42):
    if model_name == "RandomForest":
        return RandomForestClassifier(
            **params,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1
        )
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(
            **params,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")


def evaluate_with_logo(window_df, model):
    meta_cols = [
        "label",
        "positive_rows_in_window",
        "session_id",
        "chunk_id",
        "window_start_ts",
        "window_end_ts"
    ]
    feature_names = [c for c in window_df.columns if c not in meta_cols]

    X = window_df[feature_names].values
    y = window_df["label"].values
    groups = window_df["session_id"].values

    logo = LeaveOneGroupOut()
    oof_prob = np.zeros(len(y), dtype=float)

    for train_idx, test_idx in logo.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]

        y_train = y[train_idx]

        model.fit(X_train, y_train)
        oof_prob[test_idx] = model.predict_proba(X_test)[:, 1]

    threshold_rows = []
    for thr in np.arange(0.10, 0.91, 0.05):
        pred = (oof_prob >= thr).astype(int)
        threshold_rows.append({
            "threshold": float(thr),
            "accuracy": float(accuracy_score(y, pred)),
            "precision": float(precision_score(y, pred, zero_division=0)),
            "recall": float(recall_score(y, pred, zero_division=0)),
            "f1": float(f1_score(y, pred, zero_division=0)),
        })

    thr_df_local = pd.DataFrame(threshold_rows).sort_values(
        ["f1", "recall", "precision", "accuracy"],
        ascending=False
    ).reset_index(drop=True)

    best = thr_df_local.iloc[0].to_dict()
    best["n_windows"] = int(len(window_df))
    best["positive_rate"] = float(y.mean())

    return best, thr_df_local


search_results = []

for feature_set_name, selected_feature_cols in FEATURE_SET_OPTIONS.items():
    for window_size in SEARCH_SPACE["window_size"]:
        for min_pos in SEARCH_SPACE["min_positive_rows_in_window"]:
            window_df_search = build_window_dataset(
                df,
                CONFIG,
                selected_feature_cols=selected_feature_cols,
                window_size=window_size,
                min_positive_rows=min_pos
            )

            if len(window_df_search) == 0:
                continue

            for model_cfg in SEARCH_SPACE["models"]:
                model_name = model_cfg["model_name"]

                for params in model_cfg["params_list"]:
                    model = make_model(model_name, params, random_state=CONFIG["random_state"])
                    best_result, _ = evaluate_with_logo(window_df_search, model)

                    best_result["feature_set"] = feature_set_name
                    best_result["base_feature_cols"] = ", ".join(selected_feature_cols)
                    best_result["window_size"] = window_size
                    best_result["min_positive_rows_in_window"] = min_pos
                    best_result["model_name"] = model_name
                    best_result["model_params"] = json.dumps(params)

                    search_results.append(best_result)

search_df = pd.DataFrame(search_results).sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=False
).reset_index(drop=True)

search_df.head(30)

,threshold,accuracy,precision,recall,f1,n_windows,positive_rate,feature_set,base_feature_cols,window_size,min_positive_rows_in_window,model_name,model_params
0,0.25,0.821084,0.532319,0.542636,0.537428,1347,0.191537,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",9,3,ExtraTrees,"{""n_estimators"": 300, ""max_depth"": null, ""min_..."
1,0.25,0.852802,0.552632,0.520661,0.536170,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,ExtraTrees,"{""n_estimators"": 300, ""max_depth"": null, ""min_..."
2,0.30,0.812175,0.509434,0.523256,0.516252,1347,0.191537,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",9,3,ExtraTrees,"{""n_estimators"": 500, ""max_depth"": null, ""min_..."
3,0.25,0.827819,0.477193,0.561983,0.516129,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,RandomForest,"{""n_estimators"": 700, ""max_depth"": 12, ""min_sa..."
4,0.30,0.799555,0.480000,0.558140,0.516129,1347,0.191537,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",9,3,ExtraTrees,"{""n_estimators"": 700, ""max_depth"": 12, ""min_sa..."
5,0.20,0.820392,0.460784,0.582645,0.514599,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,RandomForest,"{""n_estimators"": 300, ""max_depth"": null, ""min_..."
6,0.25,0.787676,0.457576,0.585271,0.513605,1347,0.191537,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",9,3,RandomForest,"{""n_estimators"": 700, ""max_depth"": 12, ""min_sa..."
7,0.25,0.811614,0.443425,0.599174,0.509666,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,ExtraTrees,"{""n_estimators"": 500, ""max_depth"": null, ""min_..."
8,0.25,0.824443,0.468750,0.557851,0.509434,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,RandomForest,"{""n_estimators"": 500, ""max_depth"": null, ""min_..."
9,0.30,0.827144,0.474638,0.541322,0.505792,1481,0.163403,A_all_10,"ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...",7,3,ExtraTrees,"{""n_estimators"": 700, ""max_depth"": 12, ""min_sa..."


In [ ]:
best_search = search_df.iloc[0].copy()
print("Best setup found:")
print(best_search)

BEST_FEATURE_SET_NAME = best_search["feature_set"]
BEST_BASE_FEATURE_COLS = FEATURE_SET_OPTIONS[BEST_FEATURE_SET_NAME]
BEST_WINDOW_SIZE = int(best_search["window_size"])
BEST_MIN_POSITIVE_ROWS = int(best_search["min_positive_rows_in_window"])
BEST_MODEL_NAME = best_search["model_name"]
BEST_MODEL_PARAMS = json.loads(best_search["model_params"])

CONFIG["feature_cols"] = BEST_BASE_FEATURE_COLS
CONFIG["window_size"] = BEST_WINDOW_SIZE
CONFIG["min_positive_rows_in_window"] = BEST_MIN_POSITIVE_ROWS
CONFIG["model_name"] = BEST_MODEL_NAME
CONFIG["model_params"] = BEST_MODEL_PARAMS

window_df = build_window_dataset(
    df,
    CONFIG,
    selected_feature_cols=BEST_BASE_FEATURE_COLS,
    window_size=BEST_WINDOW_SIZE,
    min_positive_rows=BEST_MIN_POSITIVE_ROWS
)

print("\nUpdated CONFIG:")
print(json.dumps({
    "feature_cols": CONFIG["feature_cols"],
    "window_size": CONFIG["window_size"],
    "min_positive_rows_in_window": CONFIG["min_positive_rows_in_window"],
    "model_name": CONFIG["model_name"],
    "model_params": CONFIG["model_params"]
}, indent=2))

print("\nwindow_df shape:", window_df.shape)
print(window_df["label"].value_counts())

Best setup found:
threshold                                                                   0.25
accuracy                                                                0.821084
precision                                                               0.532319
recall                                                                  0.542636
f1                                                                      0.537428
n_windows                                                                   1347
positive_rate                                                           0.191537
feature_set                                                             A_all_10
base_feature_cols              ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, dif...
window_size                                                                    9
min_positive_rows_in_window                                                    3
model_name                                                            ExtraTrees
model_para

# Prepare X, y, groups

In [ ]:
meta_cols = [
    "label",
    "positive_rows_in_window",
    "session_id",
    "chunk_id",
    "window_start_ts",
    "window_end_ts"
]

feature_names = [c for c in window_df.columns if c not in meta_cols]

X = window_df[feature_names].values
y = window_df["label"].values
groups = window_df["session_id"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique sessions:", np.unique(groups))
print("Number of engineered features:", len(feature_names))

X shape: (1347, 100)
y shape: (1347,)
Unique sessions: ['Dataset1.csv' 'Dataset2.csv' 'Dataset3.csv' 'Dataset4.csv'
 'Dataset5.csv' 'Dataset6.csv' 'Dataset7.csv']
Number of engineered features: 100


# One session evaluate

In [ ]:
logo = LeaveOneGroupOut()

oof_prob = np.zeros(len(y), dtype=float)
per_session_results = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    test_session = groups[test_idx][0]

    clf = make_model(
        CONFIG["model_name"],
        CONFIG["model_params"],
        random_state=CONFIG["random_state"]
    )

    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    oof_prob[test_idx] = prob

    pred_05 = (prob >= 0.50).astype(int)

    per_session_results.append({
        "session_id": test_session,
        "n_windows": int(len(test_idx)),
        "positive_rate": float(y_test.mean()),
        "acc@0.50": float(accuracy_score(y_test, pred_05)),
        "precision@0.50": float(precision_score(y_test, pred_05, zero_division=0)),
        "recall@0.50": float(recall_score(y_test, pred_05, zero_division=0)),
        "f1@0.50": float(f1_score(y_test, pred_05, zero_division=0)),
    })

cv_df = pd.DataFrame(per_session_results)
cv_df

,session_id,n_windows,positive_rate,acc@0.50,precision@0.50,recall@0.50,f1@0.50
0,Dataset1.csv,223,0.000000,1.000000,0.000000,0.000000,0.000000
1,Dataset2.csv,77,0.090909,0.922078,0.666667,0.285714,0.400000
2,Dataset3.csv,157,0.286624,0.719745,1.000000,0.022222,0.043478
3,Dataset4.csv,179,0.122905,0.877095,0.000000,0.000000,0.000000
4,Dataset5.csv,64,0.078125,0.937500,1.000000,0.200000,0.333333
5,Dataset6.csv,624,0.285256,0.748397,0.838710,0.146067,0.248804
6,Dataset7.csv,23,0.043478,0.956522,0.000000,0.000000,0.000000


# Find best threshold from out-of-fold probabilities

In [ ]:
threshold_results = []

for thr in np.arange(0.10, 0.91, 0.05):
    pred = (oof_prob >= thr).astype(int)

    threshold_results.append({
        "threshold": float(thr),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
    })

thr_df = (
    pd.DataFrame(threshold_results)
    .sort_values(["f1", "recall", "precision", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

thr_df

,threshold,accuracy,precision,recall,f1
0,0.25,0.821084,0.532319,0.542636,0.537428
1,0.30,0.834447,0.588832,0.449612,0.509890
2,0.20,0.766147,0.425197,0.627907,0.507042
3,0.35,0.839644,0.647887,0.356589,0.460000
4,0.15,0.664439,0.326165,0.705426,0.446078
5,0.10,0.538976,0.267009,0.806202,0.401157
6,0.40,0.833705,0.663462,0.267442,0.381215
7,0.45,0.838159,0.777778,0.217054,0.339394
8,0.50,0.826281,0.852941,0.112403,0.198630
9,0.55,0.818114,0.882353,0.058140,0.109091


# Best threshold usage

In [ ]:
BEST_THRESHOLD = float(thr_df.iloc[0]["threshold"])
print("Best threshold selected by F1:", BEST_THRESHOLD)

final_oof_pred = (oof_prob >= BEST_THRESHOLD).astype(int)

print("\n===== FINAL CROSS-VALIDATION SUMMARY =====")
print("Accuracy :", accuracy_score(y, final_oof_pred))
print("Precision:", precision_score(y, final_oof_pred, zero_division=0))
print("Recall   :", recall_score(y, final_oof_pred, zero_division=0))
print("F1-score :", f1_score(y, final_oof_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y, final_oof_pred)
print(cm)

print("\nClassification Report:")
report_text = classification_report(
    y,
    final_oof_pred,
    target_names=["Not Fall", "Fall"],
    zero_division=0
)
print(report_text)

Best threshold selected by F1: 0.25000000000000006

===== FINAL CROSS-VALIDATION SUMMARY =====
Accuracy : 0.8210838901262064
Precision: 0.532319391634981
Recall   : 0.5426356589147286
F1-score : 0.5374280230326296

Confusion Matrix:
[[966 123]
 [118 140]]

Classification Report:
              precision    recall  f1-score   support

    Not Fall       0.89      0.89      0.89      1089
        Fall       0.53      0.54      0.54       258

    accuracy                           0.82      1347
   macro avg       0.71      0.71      0.71      1347
weighted avg       0.82      0.82      0.82      1347



# Per-session report at best threshold

In [ ]:
session_reports = []

for session_id in sorted(window_df["session_id"].unique()):
    mask = (window_df["session_id"].values == session_id)

    y_s = y[mask]
    p_s = oof_prob[mask]
    pred_s = (p_s >= BEST_THRESHOLD).astype(int)

    session_reports.append({
        "session_id": session_id,
        "n_windows": int(mask.sum()),
        "positive_rate": float(y_s.mean()),
        "accuracy": float(accuracy_score(y_s, pred_s)),
        "precision": float(precision_score(y_s, pred_s, zero_division=0)),
        "recall": float(recall_score(y_s, pred_s, zero_division=0)),
        "f1": float(f1_score(y_s, pred_s, zero_division=0))
    })

session_report_df = pd.DataFrame(session_reports)
session_report_df

,session_id,n_windows,positive_rate,accuracy,precision,recall,f1
0,Dataset1.csv,223,0.000000,0.878924,0.000000,0.000000,0.000000
1,Dataset2.csv,77,0.090909,0.818182,0.315789,0.857143,0.461538
2,Dataset3.csv,157,0.286624,0.789809,0.607143,0.755556,0.673267
3,Dataset4.csv,179,0.122905,0.899441,1.000000,0.181818,0.307692
4,Dataset5.csv,64,0.078125,0.921875,0.500000,1.000000,0.666667
5,Dataset6.csv,624,0.285256,0.772436,0.625000,0.505618,0.559006
6,Dataset7.csv,23,0.043478,0.913043,0.333333,1.000000,0.500000


# Train final model and save

In [ ]:
final_model = make_model(
    CONFIG["model_name"],
    CONFIG["model_params"],
    random_state=CONFIG["random_state"]
)


final_model.fit(X, y)

model_path = os.path.join(MODEL_DIR, "fall_rf_model.joblib")
joblib.dump(final_model, model_path)

artifact = {
    "model_type": CONFIG["model_name"],
    "model_params": CONFIG["model_params"],
    "feature_set_name": BEST_FEATURE_SET_NAME,
    "feature_names": feature_names,
    "base_feature_cols": CONFIG["feature_cols"],
    "time_col": CONFIG["time_col"],
    "gap_col": CONFIG["gap_col"],
    "label_col": CONFIG["label_col"],
    "window_size": CONFIG["window_size"],
    "stride": CONFIG["stride"],
    "max_gap_ms": CONFIG["max_gap_ms"],
    "min_positive_rows_in_window": CONFIG["min_positive_rows_in_window"],
    "best_threshold": BEST_THRESHOLD,
    "n_sessions": int(df["session_id"].nunique()),
    "n_rows_total": int(len(df)),
    "n_windows_total": int(len(window_df)),
    "label_distribution_windows": {
        "not_fall": int((y == 0).sum()),
        "fall": int((y == 1).sum())
    }
}

config_path = os.path.join(MODEL_DIR, "model_config.json")
with open(config_path, "w") as f:
    json.dump(artifact, f, indent=2)

# save reports
cv_path = os.path.join(MODEL_DIR, "cv_session_report.csv")
session_report_df.to_csv(cv_path, index=False)

thr_path = os.path.join(MODEL_DIR, "threshold_results.csv")
thr_df.to_csv(thr_path, index=False)

logo_path = os.path.join(MODEL_DIR, "logo_cv_results.csv")
cv_df.to_csv(logo_path, index=False)

search_path = os.path.join(MODEL_DIR, "search_results.csv")
search_df.to_csv(search_path, index=False)

report_path = os.path.join(MODEL_DIR, "overall_classification_report.txt")
with open(report_path, "w") as f:
    f.write("Best threshold selected by F1: " + str(BEST_THRESHOLD) + "\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(report_text)

print("Saved model to:", model_path)
print("Saved config to:", config_path)
print("Saved per-session report to:", cv_path)
print("Saved threshold table to:", thr_path)
print("Saved LOGO table to:", logo_path)
print("Saved text report to:", report_path)
print("Saved search results to:", search_path)

Saved model to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/fall_rf_model.joblib
Saved config to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/model_config.json
Saved per-session report to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/cv_session_report.csv
Saved threshold table to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/threshold_results.csv
Saved LOGO table to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/logo_cv_results.csv
Saved text report to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/overall_classification_report.txt
Saved search results to: /content/drive/MyDrive/Model_Dataset/model_artifacts_search/search_results.csv


# (Optional Usage)Inference helper for backend

In [ ]:
loaded_model = joblib.load(model_path)

with open(config_path, "r") as f:
    LOADED_CFG = json.load(f)

def make_window_features_for_inference(recent_rows_df, base_feature_cols):
    """
    recent_rows_df must contain exactly the latest window_size rows
    with columns:
    ax, ay, az, gx, gy, gz, mag_acc, mag_gyro, diff_acc, diff_gyro
    """
    if len(recent_rows_df) != LOADED_CFG["window_size"]:
        raise ValueError(f"Need exactly {LOADED_CFG['window_size']} rows")

    feats = {}

    for col in base_feature_cols:
        x = recent_rows_df[col].values.astype(float)

        feats[f"{col}_mean"] = float(np.mean(x))
        feats[f"{col}_std"] = float(np.std(x))
        feats[f"{col}_median"] = float(np.median(x))
        feats[f"{col}_min"] = float(np.min(x))
        feats[f"{col}_max"] = float(np.max(x))
        feats[f"{col}_range"] = float(np.max(x) - np.min(x))
        feats[f"{col}_absmax"] = float(np.max(np.abs(x)))
        feats[f"{col}_delta"] = float(x[-1] - x[0])
        feats[f"{col}_rms"] = float(np.sqrt(np.mean(np.square(x))))
        feats[f"{col}_iqr"] = float(np.percentile(x, 75) - np.percentile(x, 25))

    x_df = pd.DataFrame([feats])
    x_df = x_df[LOADED_CFG["feature_names"]]
    return x_df

def predict_recent_window(recent_rows_df):
    x_df = make_window_features_for_inference(
        recent_rows_df,
        base_feature_cols=LOADED_CFG["base_feature_cols"]
    )

    prob_fall = float(loaded_model.predict_proba(x_df)[0, 1])
    pred = 1 if prob_fall >= LOADED_CFG["best_threshold"] else 0

    return {
        "prediction": pred,
        "prediction_label": "Fall" if pred == 1 else "Not Fall",
        "prob_fall": prob_fall,
        "threshold": LOADED_CFG["best_threshold"]
    }